# 001 · Async and Await

> **Syntax › Async** · Python 3.14

Python's `async`/`await` keywords look almost identical to C#'s, but the machinery underneath is different: a coroutine is **cold** until awaited, and everything runs on a **single-threaded event loop** rather than the .NET thread pool.

**In this notebook you will learn:**

- how coroutines compare to C# `Task`
- `async def` and `await`, and why calling an async function does not run it
- what the `asyncio` event loop is, and how it differs from the .NET thread pool
- `asyncio.run()` in scripts vs top-level `await` in Jupyter
- `asyncio.sleep` vs `Task.Delay`, and why `time.sleep` is forbidden in async code
- IO-bound vs CPU-bound work, the GIL, and free-threaded builds
- `async for` and `async with`, briefly

## C# mental model

| C# | Python |
|---|---|
| `async Task<int> GetAsync()` | `async def get() -> int:` |
| `await GetAsync()` | `await get()` |
| `Task<T>` (hot: already running) | coroutine object (cold: does nothing until awaited) |
| thread pool + `SynchronizationContext` | one event loop on one thread (`asyncio`) |
| `static async Task Main()` | `asyncio.run(main())` |
| `await Task.Delay(100)` | `await asyncio.sleep(0.1)` (seconds, not ms) |
| `Thread.Sleep(100)` | `time.sleep(0.1)`: never in async code! |
| `IAsyncEnumerable<T>` + `await foreach` | async generator + `async for` |
| `await using` | `async with` |

Running tasks *concurrently* (`Task.WhenAll`, `Task.Run`, cancellation, timeouts) is covered in *Async › 002 · Running Tasks Concurrently*.

## `async def` and `await`

```csharp
// C#
static async Task<string> FetchUserAsync(int id)
{
    await Task.Delay(100);              // simulate a network call
    return $"user-{id}";
}

var user = await FetchUserAsync(1);
```

An `async def` function is a **coroutine function**. Inside it, `await` suspends the coroutine until the awaited operation completes, letting the event loop run other work meanwhile. Python has no `Async` suffix convention.

In [ ]:
import asyncio


async def fetch_user(user_id: int) -> str:
    await asyncio.sleep(0.1)            # simulate a network call
    return f"user-{user_id}"


user = await fetch_user(1)              # top-level await works in Jupyter
print(user)

Note the return annotation is just `-> str`, not `-> Coroutine[..., str]`: you annotate what the `await` eventually produces, much like writing `Task<string>` but listing only `string`.

`await` can only be used **inside** an `async def` (or at the top level of a Jupyter cell). Using it in a regular function is a `SyntaxError`, just as `await` in a non-`async` C# method is a compile error.

In [ ]:
try:
    compile("def regular():\n    await asyncio.sleep(1)", "<demo>", "exec")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e.msg}")

## Calling an async function does not run it

This is the biggest difference from C#. Calling a C# `async` method starts it immediately and returns a **hot** `Task`. Calling a Python `async def` function only creates a **coroutine object**; none of its body runs until something awaits it (or schedules it as a task).

```csharp
// C#
var task = DoWorkAsync();   // DoWorkAsync is ALREADY running here
await task;
```

In [ ]:
async def do_work():
    print("  ...body is running")
    return 42


coro = do_work()                # nothing printed: the body has not started
print("created:", coro)
print("type:", type(coro).__name__)

result = await coro             # NOW the body runs
print("result:", result)

⚠️ **Gotcha:** forgetting `await` is a silent bug. You get a coroutine object instead of the result, the work never happens, and Python only emits a `RuntimeWarning: coroutine '...' was never awaited` when the object is garbage-collected. (Here we call `.close()` to discard it cleanly.)

In [ ]:
async def save_order(order_id):
    print(f"  saving order {order_id}")
    return True


ok = save_order(7)              # forgot await!
print(bool(ok), ok)             # truthy coroutine object, nothing was saved
ok.close()

⚠️ **Gotcha:** a coroutine can be awaited **only once**. A C# `Task` can be awaited any number of times and caches its result; a bare coroutine cannot. Wrap it in a task *(see Async › 002 · Running Tasks Concurrently)* if several places need the result.

In [ ]:
coro = fetch_user(2)
print(await coro)
try:
    await coro
except RuntimeError as e:
    print(f"{type(e).__name__}: {e}")

## Exceptions propagate through `await`

Just like C#, an exception raised inside a coroutine surfaces at the `await`, and you handle it with a normal `try`/`except`.

In [ ]:
async def load_config(name):
    await asyncio.sleep(0.01)
    raise FileNotFoundError(f"no config named {name!r}")


try:
    await load_config("prod")
except FileNotFoundError as e:
    print(f"{type(e).__name__}: {e}")

## The event loop: one thread, cooperative multitasking

`asyncio` runs all coroutines on a **single thread** using an event loop. A coroutine runs until it hits an `await` that has to wait (a timer, a socket, ...), then hands control back to the loop, which resumes whichever coroutine is ready next. Nothing is preempted: switches happen **only at `await` points**.

In .NET, continuations after `await` are typically scheduled on the **thread pool** (in console/ASP.NET Core apps), so your code may resume on a different thread and genuinely run in parallel with other tasks.

In [ ]:
import threading


async def where_am_i(label):
    await asyncio.sleep(0.01)
    return f"{label} ran on thread {threading.current_thread().name!r}"


print(await where_am_i("first"))
print(await where_am_i("second"))
print("running loop:", type(asyncio.get_running_loop()).__name__)

A side effect of the single-threaded model: code **between** two `await`s can never be interrupted by another coroutine, so you rarely need locks for simple shared state (you still do if the critical section itself contains an `await`).

## `asyncio.run()` in scripts vs top-level `await` in Jupyter

In a normal `.py` script there is no running event loop. You create one with `asyncio.run()`, which plays the role of `static async Task Main()`:

```python
# main.py
import asyncio

async def main() -> None:
    user = await fetch_user(1)
    print(user)

if __name__ == "__main__":
    asyncio.run(main())    # creates the loop, runs main() to completion, closes the loop
```

Jupyter (IPython) already runs an event loop for you and allows `await` directly in a cell, which is why this notebook never calls `asyncio.run()`. Calling it inside Jupyter fails with `RuntimeError: asyncio.run() cannot be called from a running event loop`.

⚠️ **Gotcha:** top-level `await` is a **Jupyter/IPython feature** (and of the `python -m asyncio` REPL). In a plain `.py` file, `await` outside an `async def` is a `SyntaxError`.

## `asyncio.sleep` vs `Task.Delay`

`asyncio.sleep(seconds)` is the non-blocking delay. Note the unit: **seconds** (a float), not milliseconds. Awaiting coroutines one after another runs them **sequentially**, exactly as consecutive `await`s do in C#:

```csharp
// C#
await Task.Delay(100);
await Task.Delay(100);   // ~200 ms in total
```

In [ ]:
import time

start = time.perf_counter()
await asyncio.sleep(0.1)
await asyncio.sleep(0.1)
print(f"sequential: {time.perf_counter() - start:.2f}s")    # ~0.20s

To overlap them you need tasks (`asyncio.gather`, `asyncio.TaskGroup`, ...), the subject of *Async › 002 · Running Tasks Concurrently*.

## ⚠️ Never use `time.sleep` (or other blocking calls) in async code

`time.sleep` blocks the **whole thread**, and since the event loop has only one thread, every other coroutine freezes too. In C#, `Thread.Sleep` in async code wastes a thread-pool thread; in Python it stalls everything.

The demo below runs a "heartbeat" in the background while another coroutine waits for 0.3 s, first with `asyncio.sleep`, then with `time.sleep`. (`create_task` schedules the heartbeat to run concurrently; it is explained in the next notebook.)

In [ ]:
async def heartbeat(beats):
    for _ in range(beats):
        print("    ♥ tick")
        await asyncio.sleep(0.1)


async def wait_politely():
    await asyncio.sleep(0.3)        # yields to the loop: heartbeat keeps ticking


async def wait_rudely():
    time.sleep(0.3)                 # blocks the loop: heartbeat is frozen


for waiter in (wait_politely, wait_rudely):
    print(waiter.__name__)
    beat = asyncio.create_task(heartbeat(3))
    await waiter()
    print("  waiter done")
    await beat

With `asyncio.sleep` the ticks interleave with the wait; with `time.sleep` all the ticks only happen **after** the blocking call returns. The same applies to any blocking API: `requests.get`, synchronous database drivers, file I/O on slow disks, heavy computation. Use async libraries, or offload blocking calls with `asyncio.to_thread` *(see Async › 002 · Running Tasks Concurrently)*.

## IO-bound vs CPU-bound work

`asyncio` helps with **IO-bound** work: lots of waiting on networks, disks, or timers. It does nothing for **CPU-bound** work, because only one coroutine runs at a time.

| Workload | C# | Python |
|---|---|---|
| Many concurrent I/O waits | `async`/`await` | `asyncio` |
| Blocking library call | `Task.Run(...)` | `asyncio.to_thread(...)` (a thread) |
| CPU-heavy parallel work | `Parallel.For`, `Task.Run` | `concurrent.futures.ProcessPoolExecutor` (processes) |

The reason for the last row is the **GIL** (Global Interpreter Lock): in the standard CPython build, only one thread executes Python bytecode at a time, so threads do not speed up pure-Python CPU work. Separate processes each have their own interpreter and GIL.

Python 3.13 introduced an experimental **free-threaded** build (PEP 703) that can run without the GIL, and in 3.14 that build is officially supported, but it is still a **separate, optional build** (often installed as `python3.14t`), not the default. Check which one you are running:

In [ ]:
import sys
import sysconfig

print("free-threaded build:", bool(sysconfig.get_config_var("Py_GIL_DISABLED")))
print("GIL currently enabled:", sys._is_gil_enabled())   # 3.13+, private helper

## `async for`: asynchronous iteration

An `async def` that contains `yield` is an **async generator**, the equivalent of a C# method returning `IAsyncEnumerable<T>`. Consume it with `async for` (C#: `await foreach`). Async comprehensions work too *(see Iterators & Generators › 002 · Generators)*.

```csharp
// C#
static async IAsyncEnumerable<int> TickAsync(int count)
{
    for (var i = 0; i < count; i++)
    {
        await Task.Delay(10);
        yield return i;
    }
}

await foreach (var tick in TickAsync(3)) Console.WriteLine(tick);
```

In [ ]:
async def tick(count):
    for i in range(count):
        await asyncio.sleep(0.01)
        yield i


async for t in tick(3):
    print("tick", t)

squares = [t * t async for t in tick(4)]     # async comprehension
print(squares)

## `async with`: asynchronous context managers

`async with` is the async version of `with` *(see Error Handling › 003 · Context Managers)*, the equivalent of C#'s `await using` with `IAsyncDisposable`. The easiest way to write one is `contextlib.asynccontextmanager`:

```csharp
// C#
await using var conn = await OpenConnectionAsync();
```

In [ ]:
from contextlib import asynccontextmanager


@asynccontextmanager
async def open_connection(name):
    print(f"  opening {name}")
    await asyncio.sleep(0.01)
    try:
        yield f"<conn {name}>"
    finally:
        await asyncio.sleep(0.01)        # async cleanup, like DisposeAsync
        print(f"  closed {name}")


async with open_connection("db") as conn:
    print("  using", conn)

## Summary

| Concept | C# | Python |
|---|---|---|
| Async function | `async Task<T> FooAsync()` | `async def foo() -> T` |
| Calling it | starts running (hot `Task`) | creates a coroutine (cold) |
| Await | `await FooAsync()` | `await foo()` |
| Await twice | allowed | `RuntimeError` for a bare coroutine |
| Scheduler | thread pool | single-threaded event loop |
| Entry point | `async Task Main()` | `asyncio.run(main())` |
| Top-level `await` | top-level statements, C# scripts | Jupyter cells only (and the `python -m asyncio` REPL) |
| Non-blocking delay | `Task.Delay(ms)` | `asyncio.sleep(seconds)` |
| Blocking delay | `Thread.Sleep` | `time.sleep`: avoid in async code |
| Async stream | `IAsyncEnumerable<T>` / `await foreach` | async generator / `async for` |
| Async dispose | `await using` | `async with` |

## Exercises

1. Write `async def countdown(n)` that prints `n, n-1, ..., 1` with a 0.05 s pause between numbers, then await it.
2. What is printed by `r = fetch_user(5); print(type(r).__name__)`? How do you get the actual user string?
3. Write an async generator `paginate(items, size)` that yields lists of `size` items with a short `asyncio.sleep` between pages (simulating an API), and collect the pages with an async comprehension.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import asyncio

# 1
async def countdown(n):
    for i in range(n, 0, -1):
        print(i)
        await asyncio.sleep(0.05)

await countdown(3)

# 2 — prints "coroutine": calling an async function only creates a coroutine.
#     Get the value with:  user = await fetch_user(5)
#     (and call r.close() if you are discarding r without awaiting it)

# 3
async def paginate(items, size):
    for start in range(0, len(items), size):
        await asyncio.sleep(0.01)
        yield items[start:start + size]

pages = [page async for page in paginate(list(range(7)), 3)]
print(pages)   # [[0, 1, 2], [3, 4, 5], [6]]
```

</details>